# 모듈 (2-1) LangSmith 실전 핸즈온 — 실제 트레이싱·데이터셋·평가
## 에이전트 거버넌스 (Governance) · 관측성(Observability)

---

### 학습 목표
1. **LangSmith 계정·API 키**를 발급하고 `.env` 로 추적을 켠다
2. LangChain 호출이 **자동으로 트레이스**되는 것을 실제 대시보드에서 확인한다
3. `@traceable` 로 **임의의 파이썬 함수**(RAG 파이프라인)를 계층형 트레이스로 남긴다
4. **데이터셋**을 만들고 **`evaluate()`** 로 규칙 기반 + LLM-as-judge 평가를 돌린다
5. **메타데이터·태그·피드백**으로 실행을 분류·평가하고, LangSmith **UI** 를 둘러본다

> 📦 **환경 설치·실행 명령**은 [`env_guides/M03_2_1_langsmith_hands_on.md`](env_guides/M03_2_1_langsmith_hands_on.md) 에 정리되어 있습니다
> (Ollama 준비, `LANGSMITH_API_KEY` 발급·설정).

> 이 노트북은 [`(2) LangSmith 트레이싱`](M03_2_tracing.ipynb) 의 **시뮬레이션 대시보드**를 넘어,
> **실제 LangSmith 서비스**([smith.langchain.com](https://smith.langchain.com))에 직접 붙어서 트레이스·데이터셋·평가를 다룹니다.

> **LLM 은 `.env` 의 `LLM_PROVIDER` 를 그대로 사용**합니다(`utils.get_llm()`). 기본은 로컬 Ollama(`qwen3:8b`)이며,
> LangSmith 추적은 LangChain 계층에서 동작하므로 **어떤 공급자를 쓰든 동일하게 기록**됩니다.

---

## 0. 공통 셋업

`.env` 를 재로드하고 기본 LLM(`utils.get_llm()`)과 LangSmith SDK 를 준비합니다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가
import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER / LANGSMITH_* 갱신) + 공급자 상태 출력

# 공통 응답 정규화 헬퍼 (공급자 무관 + qwen3 의 <think> 제거)
from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text, invoke_text

# 이 노트북이 쓰는 추가 의존성 (LangSmith SDK 포함) — uv → 실패 시 pip
utils.uv_install(['langsmith', 'langchain', 'langchain-core', 'langchain-openai'])

llm = utils.get_llm()  # .env 의 LLM_PROVIDER 사용 (기본 ollama/qwen3:8b)
print('LLM 공급자:', utils.LLM_PROVIDER)

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langsmith', 'langchain', 'langchain-core', 'langchain-openai']
LLM 공급자: nvidia


## 1. LangSmith 란?

**LangSmith** 는 LangChain 팀이 만든 **LLM 애플리케이션 관측·테스트·모니터링 플랫폼**입니다.
웹 개발의 브라우저 DevTools 처럼, *LLM 앱 내부에서 무슨 일이 일어나는지*를 그대로 보여줍니다.
LangChain 을 쓰지 않는 앱에서도 사용할 수 있습니다.

| 기능 | 설명 |
|---|---|
| **Tracing(추적)** | LLM 앱의 모든 단계(프롬프트·도구 호출·응답)를 계층형 트레이스로 기록 |
| **Debugging(디버깅)** | 어디서 무엇이 잘못됐는지, 정확히 어떤 프롬프트가 나갔는지 확인 |
| **Testing/Evaluation(평가)** | 데이터셋을 만들고 앱 성능을 체계적으로 채점·비교 |
| **Monitoring(모니터링)** | 프로덕션에서 지연·토큰·비용·오류율을 추적하고 알림 설정 |
| **Prompt Management(프롬프트 관리)** | 프롬프트 버전 관리·Playground·A/B 테스트 |

> **왜 필요한가?** 사용자가 "답이 이상하다"고 할 때, 추적이 없으면 원인을 알 수 없습니다.
> LangSmith 는 모든 상호작용을 추적해 **토큰 사용량·지연 시간**을 보여주고 **프롬프트 버전을 비교**하게 해줍니다.
> *측정할 수 없으면 개선할 수 없습니다.*

## 2. 계정·API 키 발급 & `.env` 설정

1. [smith.langchain.com](https://smith.langchain.com) 에 가입합니다(무료 티어 제공).
2. **Settings → API Keys → Create API Key** 로 키를 발급받아 복사합니다.
3. `notebooks/.env` 에 아래 항목을 채웁니다(`.env.example` 복사).

```ini
# notebooks/.env
LANGSMITH_API_KEY=lsv2_...            # 위에서 발급한 키
LANGSMITH_PROJECT=agentic-ai-tutorial # 트레이스를 묶을 프로젝트 이름(자유)
# (선택) 자체 호스팅/리전 엔드포인트를 쓸 때만 변경
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
```

> 키가 없어도 아래 셀들은 **오류 없이** 넘어가도록 만들어 두었습니다(추적/데이터셋/평가만 건너뜀).
> 다만 이 노트북의 목적은 *실제 LangSmith 사용*이므로, 키를 넣고 재실행하는 것을 권장합니다.

In [2]:
# LangSmith 추적을 켠다. 최신 이름(LANGSMITH_*)과 구버전 별칭(LANGCHAIN_*)을 함께 설정해 호환성을 확보.
import os

LANGSMITH_API_KEY  = os.getenv('LANGSMITH_API_KEY', '')
LANGSMITH_PROJECT  = os.getenv('LANGSMITH_PROJECT', 'agentic-ai-tutorial')
LANGSMITH_ENDPOINT = os.getenv('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')

LANGSMITH_ENABLED = bool(LANGSMITH_API_KEY)  # 이후 셀에서 실제 전송 여부를 가르는 플래그

if LANGSMITH_ENABLED:
    for k, v in {
        'LANGSMITH_TRACING': 'true',   'LANGCHAIN_TRACING_V2': 'true',
        'LANGSMITH_ENDPOINT': LANGSMITH_ENDPOINT, 'LANGCHAIN_ENDPOINT': LANGSMITH_ENDPOINT,
        'LANGSMITH_API_KEY': LANGSMITH_API_KEY,   'LANGCHAIN_API_KEY': LANGSMITH_API_KEY,
        'LANGSMITH_PROJECT': LANGSMITH_PROJECT,   'LANGCHAIN_PROJECT': LANGSMITH_PROJECT,
    }.items():
        os.environ[k] = v
    print(f"✅ LangSmith 추적 활성화 — 프로젝트 '{LANGSMITH_PROJECT}'")
    print(f"   대시보드: https://smith.langchain.com  →  Projects  →  {LANGSMITH_PROJECT}")
else:
    print("⚠️  LANGSMITH_API_KEY 가 없습니다. smith.langchain.com 에서 무료 키를 발급해")
    print("    notebooks/.env 의 LANGSMITH_API_KEY 에 넣고 이 노트북을 재실행하세요.")
    print("    (키 없이도 모든 셀은 오류 없이 진행됩니다 — 추적/데이터셋/평가만 건너뜀)")

✅ LangSmith 추적 활성화 — 프로젝트 'agentic-ai-tutorial'
   대시보드: https://smith.langchain.com  →  Projects  →  agentic-ai-tutorial


In [3]:
# LangSmith 클라이언트 준비 + 연결 확인 (키가 있을 때만)
client = None
if LANGSMITH_ENABLED:
    from langsmith import Client
    client = Client()  # 환경변수 LANGSMITH_API_KEY / _ENDPOINT 를 자동으로 읽음
    try:
        # 워크스페이스 접근이 되는지 가볍게 확인
        projects = list(client.list_projects(limit=1))
        print('LangSmith 연결 OK — 클라이언트 준비 완료')
    except Exception as e:
        print('LangSmith 연결 확인 실패(키/네트워크 확인):', e)

LangSmith 연결 OK — 클라이언트 준비 완료


## 3. 첫 트레이스 — 자동 추적

추적이 켜져 있으면 **모든 LangChain 호출이 자동으로 트레이스**됩니다. 별도 코드가 필요 없습니다.
아래 한 번의 `llm.invoke(...)` 가 프로젝트에 하나의 트레이스로 남습니다.

In [4]:
from langchain_core.messages import HumanMessage

resp = llm.invoke([HumanMessage(content='LangSmith 를 한 문장으로 설명해줘.')])
print(to_text(resp.content))

if LANGSMITH_ENABLED:
    print(f"\n→ smith.langchain.com 의 '{LANGSMITH_PROJECT}' 프로젝트에서")
    print("  방금 호출의 입력/출력/토큰/지연 시간을 트레이스로 확인하세요.")

LangSmith는 인공지능 기반의 언어 학습 플랫폼으로, 사용자가 원하는 언어를 학습할 수 있도록 개인화된 학습 계획과 자막, 자막 번역, 음성 번역, 언어 학습 게임 등 다양한 기능을 제공합니다.

→ smith.langchain.com 의 'agentic-ai-tutorial' 프로젝트에서
  방금 호출의 입력/출력/토큰/지연 시간을 트레이스로 확인하세요.


### LCEL 체인 — 중첩 트레이스

`prompt | llm | parser` 처럼 여러 단계를 이으면(LCEL), 트레이스에도
**ChatPromptTemplate → ChatModel → StrOutputParser** 3단계가 트리 구조로 남습니다.
어느 단계가 느린지·무엇이 들어가고 나왔는지 한눈에 보입니다.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template('{topic} 에 대한 짧은 농담을 한국어로 하나 만들어줘.')
parser = StrOutputParser()
chain = prompt | llm | parser        # LCEL 파이프라인 (각 단계가 트레이스의 자식 노드)

joke = chain.invoke({'topic': '프로그래밍'})
print(bootstrap.strip_think(joke))   # qwen3 의 <think> 제거 후 출력

프로그래밍은 언어를 배울 때와 비슷합니다. 

1. 처음에는 이해가 안 가서 '어?'를 외칩니다.
2. 몇 번 반복하면 '아하!'를 외칩니다.
3. 그 다음에는 '왜?'를 물어보는 중입니다.


## 4. `@traceable` — 임의 파이썬 함수 추적

LangChain 컴포넌트가 아닌 **일반 파이썬 함수**도 `@traceable` 데코레이터 하나로 트레이스에 넣을 수 있습니다.
아래는 미니 RAG(문서 검색 → LLM 생성) 파이프라인입니다.
`rag_answer`(부모) 안에서 `retrieve`(자식)와 LLM 호출(자식)이 **하나의 계층형 트레이스**로 묶입니다.

In [6]:
from langsmith import traceable

# 아주 단순한 문서 집합 (미니 RAG 시뮬레이션)
DOCS = {
    'ronaldo':   '크리스티아누 호날두는 포르투갈 출신 축구 선수로, 포지션은 공격수다.',
    'langsmith': 'LangSmith 는 LLM 앱의 관측·테스트·모니터링 플랫폼으로, LangChain 팀이 만들었다.',
}

@traceable(run_type='retriever', name='retrieve')
def retrieve(query: str) -> str:
    """질의에 맞는 문서를 키워드 매칭으로 아주 단순하게 고른다."""
    for key, text in DOCS.items():
        if key in query.lower():
            return text
    return '관련 문서를 찾지 못했습니다.'

@traceable(run_type='chain', name='rag_answer')
def rag_answer(query: str) -> str:
    """retrieve → LLM 생성의 2단계를 하나의 트레이스로 묶는다."""
    context = retrieve(query)                      # 자식 트레이스(retriever)
    msg = (f"다음 문맥만 근거로 질문에 한국어로 간단히 답해줘.\n"
           f"[문맥]\n{context}\n[질문]\n{query}")
    return invoke_text(llm, msg)                   # 자식 트레이스(ChatModel)

print(rag_answer('langsmith 는 무엇인가?'))
# 트레이스 계층: rag_answer(부모) → retrieve(자식) → ChatModel(자식)

LangSmith는 LLM(대규모 언어 모델) 앱의 관측, 테스트, 모니터링 플랫폼입니다.


## 5. 데이터셋 & 테스트

**데이터셋**은 입력–출력 쌍(예제)의 모음으로, 앱을 반복적으로 채점하는 **테스트 케이스**입니다.
같은 이름이 이미 있으면 재사용해 중복 생성을 피합니다.

In [9]:
DATASET_NAME = 'langsmith-hands-on-qa'

# 데이터셋 예제: question → 모범답안(answer)
EXAMPLES = [
    {'inputs': {'question': 'langsmith 는 무엇인가?'},
     'outputs': {'answer': 'LLM 앱의 관측·테스트·모니터링 플랫폼'}},
    {'inputs': {'question': 'ronaldo 의 포지션은?'},
     'outputs': {'answer': '공격수'}},
]

dataset = None
if LANGSMITH_ENABLED:
    if client.has_dataset(dataset_name=DATASET_NAME):
        dataset = client.read_dataset(dataset_name=DATASET_NAME)
        print(f"기존 데이터셋 재사용: {DATASET_NAME}")
    else:
        dataset = client.create_dataset(
            dataset_name=DATASET_NAME,
            description='LangSmith 핸즈온용 미니 QA 데이터셋',
        )
        client.create_examples(
            inputs=[e['inputs'] for e in EXAMPLES],
            outputs=[e['outputs'] for e in EXAMPLES],
            dataset_id=dataset.id,
        )
        print(f"데이터셋 생성 완료: {DATASET_NAME} (예제 {len(EXAMPLES)}개)")
    print("→ LangSmith 의 Datasets & Experiments 탭에서 확인하세요.")
else:
    print("(건너뜀) LANGSMITH_API_KEY 가 있어야 데이터셋을 만들 수 있습니다.")

기존 데이터셋 재사용: langsmith-hands-on-qa
→ LangSmith 의 Datasets & Experiments 탭에서 확인하세요.


## 6. 평가(Evaluation) — `evaluate()`

LangSmith 최신 `evaluate()` 는 **데이터셋의 각 예제**에 대해 대상 함수를 실행하고,
지정한 **평가자(evaluator)** 들로 결과를 채점해 하나의 **실험(experiment)** 으로 기록합니다.

- **평가 대상(target)**: `inputs` 를 받아 `outputs` 를 돌려주는 함수 → 여기서는 앞의 `rag_answer`
- **평가자 1(규칙 기반)**: "모르겠다" 류 회피 답변을 감점 (참고자료의 `check_not_idk` 를 현대 시그니처로)
- **평가자 2(LLM-as-judge)**: 모범답안과 의미가 맞는지 **LLM 이 채점**

In [7]:
# 평가 대상: 데이터셋 예제의 inputs(dict)를 받아 outputs(dict)를 돌려준다.
def qa_target(inputs: dict) -> dict:
    """예제의 question 을 RAG 파이프라인에 넣어 답을 생성한다."""
    return {'answer': rag_answer(inputs['question'])}

# 평가자 1 — 규칙 기반: 회피성 답변이면 0점
def not_uncertain(outputs: dict) -> dict:
    ans = (outputs or {}).get('answer', '')
    ok = ('모르' not in ans) and ('찾지 못' not in ans)
    return {'key': 'not_uncertain', 'score': 1 if ok else 0}

# 평가자 2 — LLM-as-judge: 모범답안(reference)과 의미 일치 여부를 LLM 이 0/1 로 채점
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    judge = (f"질문: {inputs['question']}\n"
             f"모범답안: {reference_outputs['answer']}\n"
             f"제출답안: {outputs['answer']}\n"
             f"제출답안이 모범답안과 의미상 일치하면 1, 아니면 0. 숫자 하나만 답해줘.")
    verdict = invoke_text(llm, judge)
    return {'key': 'correctness', 'score': 1 if '1' in verdict[:5] else 0}

In [10]:
if LANGSMITH_ENABLED and dataset is not None:
    try:
        from langsmith import evaluate           # 최신 위치
    except ImportError:
        from langsmith.evaluation import evaluate # 구버전 폴백

    results = evaluate(
        qa_target,
        data=DATASET_NAME,
        evaluators=[not_uncertain, correctness],
        experiment_prefix='qa-hands-on',
        metadata={'provider': utils.LLM_PROVIDER},  # 실험 파라미터를 함께 기록
    )
    print('평가 완료 — LangSmith 의 Datasets & Experiments 탭에서 예제별 점수를 확인하세요.')
else:
    # 키가 없어도 평가 '로직' 자체는 로컬에서 시연 (LangSmith 기록은 없음)
    print('(로컬 시연) LangSmith 없이 평가 로직만 실행:')
    for ex in EXAMPLES:
        out = qa_target(ex['inputs'])
        r1 = not_uncertain(out)
        r2 = correctness(ex['inputs'], out, ex['outputs'])
        print(f"  Q: {ex['inputs']['question']}")
        print(f"     답: {out['answer'][:50]}...")
        print(f"     {r1}, {r2}")

View the evaluation results for experiment: 'qa-hands-on-af250719' at:
https://smith.langchain.com/o/b8901547-5702-4968-9ba6-406c7fa1833c/datasets/8b68566f-cfa9-49fc-b0b9-8fc661acd9c2/compare?selectedSessions=4ae47f74-19f0-4794-8270-edf2a455337f




0it [00:00, ?it/s]

평가 완료 — LangSmith 의 Datasets & Experiments 탭에서 예제별 점수를 확인하세요.


## 7. 메타데이터·태그 & 피드백

- **태그·메타데이터**를 붙이면 UI 에서 실행을 **필터링·그룹화**하기 쉽습니다(버전·사용자·환경별 비교).
- **피드백(feedback)** 은 특정 실행에 점수·코멘트를 남기는 것으로, 프로덕션 품질 추적의 핵심입니다.
  사람 평가(👍/👎)나 자동 평가 점수를 실행에 연결할 수 있습니다.

In [11]:
from langchain_core.messages import HumanMessage

# run_name / tags / metadata 를 config 로 넘기면 트레이스에 그대로 기록된다.
resp = llm.invoke(
    [HumanMessage(content='LangSmith 의 장점 3가지를 한 줄로 요약해줘.')],
    config={
        'run_name': 'tagged-demo',
        'tags': ['hands-on', 'M03_2_1'],
        'metadata': {'lesson': 'M03_2_1', 'provider': utils.LLM_PROVIDER},
    },
)
print(to_text(resp.content))

LangSmith는 다음과 같은 장점을 가지고 있습니다.

1. **자연어 처리**: LangSmith는 자연어 처리를 위한 강력한 도구를 제공하여 언어 모델링, 문법 분석, 의미 추론 등 다양한 자연어 처리 작업을 지원합니다.
2. **고유의 언어 모델**: LangSmith는 고유의 언어 모델을 제공하여 사용자가 원하는 언어와 문법을 사용할 수 있도록 지원합니다.
3. **고성능**: LangSmith는 고성능을 제공하여 빠른 처리 속도와 높은 정확도를 지원하여 사용자가 원하는 결과를 얻을 수 있도록 합니다.


In [12]:
# 방금 실행에 사람 피드백(점수)을 남긴다. collect_runs 로 실행 id 를 안정적으로 얻는다.
if LANGSMITH_ENABLED:
    from langchain_core.tracers.context import collect_runs

    with collect_runs() as cb:
        llm.invoke([HumanMessage(content='피드백 데모용 한 줄 답변.')])
    run_id = cb.traced_runs[0].id           # 방금 호출의 run id

    client.create_feedback(
        run_id,
        key='user_score',                   # 피드백 항목 이름
        score=1.0,                          # 0.0 ~ 1.0
        comment='핸즈온 데모 — 사람 피드백 예시',
    )
    print(f'피드백 기록 완료 (run={run_id}) — 트레이스 상세에서 Feedback 을 확인하세요.')
else:
    print('(건너뜀) 피드백 기록에는 LANGSMITH_API_KEY 가 필요합니다.')

피드백 기록 완료 (run=019f9954-4d8e-7a10-866f-817a78441658) — 트레이스 상세에서 Feedback 을 확인하세요.


## 8. LangSmith UI 둘러보기

코드로 보낸 데이터는 웹 UI 에서 확인·활용합니다. 핵심 탭은 다음과 같습니다.

| UI 영역 | 하는 일 |
|---|---|
| **Projects → Traces** | 실행 트레이스 목록. 각 트레이스를 열면 프롬프트·응답·토큰·지연·오류를 단계별로 확인 |
| **Datasets & Experiments** | 데이터셋과 평가 실험 결과. 예제별 점수, 실험(모델/프롬프트) 간 **비교** |
| **Prompt Hub & Playground** | 프롬프트를 버전 관리하고, 온도·모델을 바꿔가며 **실시간 실험**·A/B 비교 후 저장 |
| **Annotation Queues** | 사람이 출력 품질을 체계적으로 라벨링 → 골드 데이터셋 구축(파인튜닝·회귀 테스트용) |
| **Monitoring / Dashboards** | 프로덕션에서 지연·토큰·비용·오류율·피드백 점수를 추적하고 **알림(Alerts)** 설정 |

> **개발 vs 프로덕션**: 개발 단계에서는 *디버깅·데이터셋 테스트*에 쓰고,
> 프로덕션에서는 *성능·비용 모니터링과 사용자 피드백 수집*으로 앱을 지속 개선합니다.

> **멀티에이전트/그래프 추적**: LangGraph 로 만든 멀티에이전트도 추적을 켜 두면
> 슈퍼바이저·각 에이전트·도구 호출이 **하나의 트레이스**로 자동 기록됩니다.
> (이 강의 [`M02_6 LangGraph 멀티에이전트`](M02_6_langgraph_multiagent.ipynb) 를 실행하면 동일하게 트레이스가 남습니다.)

---
## 정리

1. **자동 추적**: `LANGSMITH_TRACING=true` + API 키만 있으면 모든 LangChain 호출이 트레이스로 기록된다
2. **`@traceable`**: LangChain 이 아닌 일반 함수도 계층형 트레이스에 넣을 수 있다
3. **데이터셋 + `evaluate()`**: 입력–출력 예제로 규칙 기반 + LLM-as-judge 평가를 자동화한다
4. **메타데이터·태그·피드백**: 실행을 분류·비교하고 품질 점수를 남긴다
5. **UI**: Traces·Experiments·Playground·Monitoring 으로 디버깅부터 프로덕션 모니터링까지 커버

### 실전 팁 (참고자료 공통 강조)
- **추적은 항상 켜라** — 무료이고 디버깅 시간을 크게 줄여준다
- **데이터셋을 일찍 만들어라** — 회귀 테스트·개선 비교의 기준이 된다
- **메타데이터로 태깅하라** — 버전·사용자·환경별 필터링이 쉬워진다
- **토큰 사용량을 모니터링하라** — LLM 비용은 빠르게 늘어난다

### 다음 노트북
- **(3) Self-Refine:** 품질 기준 미달 시 자기 비평 → 반복 개선

### 참고 자료
- LangSmith 문서: https://docs.smith.langchain.com
- 평가(Evaluation) 가이드: https://docs.smith.langchain.com/evaluation